# Advance chain example:
1. __Multi-format Data Extraction Chain__   
2. __Batch Processing - Efficient handling of multiple items__

This is optional in case users wants to explore more advance cases can use this

In [2]:
# check langchain version it should be 0.3 if not uncomment below pip command to install the correct version
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.3.23
0.3.21


In [3]:
#pip uninstall langchain -y && pip uninstall langchain-core -y && pip uninstall langchain-community -y
# !pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24


## 1. Multi-format Data Extraction Chain

In [4]:
import re
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Load environment variables
load_dotenv()

# Create model
model = ChatOpenAI(model="gpt-4o-mini")

/Users/goura/Desktop/Workspace/Agentic AI/py313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel



In [6]:

# Chain for extracting structured information from unstructured text
extraction_prompt = ChatPromptTemplate.from_template("""
Extract the following information from the job description below and return as valid JSON:

Required fields:
- company_name
- position  
- requirements (as a list)
- benefits (as a list)
- contact_info

Job Description: {job_description}

Return ONLY valid JSON without any additional text:
""")

validation_prompt = ChatPromptTemplate.from_template("""
Validate and clean this extracted job data. Fix any errors and ensure all fields are properly formatted.
If any field is missing, add it with value "Not specified".

Raw extraction: {raw_extraction}

Return ONLY valid JSON without any additional text:
""")

# Create the extraction chains
extraction_chain = extraction_prompt | model | StrOutputParser()
validation_chain = validation_prompt | model | StrOutputParser()

# Helper function to extract JSON from text
def extract_json_from_text(text):
    """Extract JSON from text response"""
    try:
        # Try to parse directly first
        return json.loads(text)
    except json.JSONDecodeError:
        # Try to find JSON pattern in text
        json_match = re.search(r'\{[^{}]*\{[^{}]*\}[^{}]*\}|\{[^{}]*\}', text, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group())
            except json.JSONDecodeError:
                pass
    
    # Return default structure if parsing fails
    return {
        "company_name": "Not specified",
        "position": "Not specified", 
        "requirements": ["Not specified"],
        "benefits": ["Not specified"],
        "contact_info": "Not specified"
    }

# Proper chain composition
def process_extraction(job_description):
    """Process extraction step by step"""
    # Step 1: Extract raw data
    raw_extraction = extraction_chain.invoke({"job_description": job_description})
    print(f"Raw extraction: {raw_extraction}")
    
    # Step 2: Validate and clean
    cleaned_extraction = validation_chain.invoke({"raw_extraction": raw_extraction})
    print(f"Cleaned extraction: {cleaned_extraction}")
    
    # Step 3: Parse to JSON
    json_data = extract_json_from_text(cleaned_extraction)
    
    return json_data

# Simple working chain
extraction_pipeline = RunnableLambda(process_extraction)



In [7]:
# Test with sample job description
job_desc = """
Senior Software Engineer at EmeretusInc. 
Requirements: 5+ years Python, AWS experience, Kubernetes, Docker.
Benefits: Health insurance, 401k matching, flexible hours, remote work options.
Contact: careers@emeretus.com or call 555-0123.
"""

print("Extracting job information...")
try:
    result = extraction_pipeline.invoke(job_desc)
    print("\nExtraction completed successfully!")
    print("\nExtracted Job Data:")
    print(json.dumps(result, indent=2))
except Exception as e:
    print(f"Error in extraction: {e}")
    # Fallback result
    result = {
        "company_name": "Emeretus Inc.",
        "position": "Senior Software Engineer",
        "requirements": ["5+ years Python", "AWS experience", "Kubernetes", "Docker"],
        "benefits": ["Health insurance", "401k matching", "flexible hours", "remote work options"],
        "contact_info": "careers@emeretus.com or call 555-0123"
    }

# Format the output nicely
formatting_prompt = ChatPromptTemplate.from_template("""
Convert this JSON job data into a well-formatted job posting:

JSON Data:
{json_data}

Create a professional job posting with clear sections for:
- Job Title and Company
- Position Overview  
- Requirements
- Benefits
- Contact Information

Formatted Job Posting:
""")

formatting_chain = formatting_prompt | model | StrOutputParser()

print("\n" + "="*50)
print("FORMATTED JOB POSTING")
print("="*50)

try:
    formatted_result = formatting_chain.invoke({"json_data": json.dumps(result, indent=2)})
    print(formatted_result)
except Exception as e:
    print(f"Error in formatting: {e}")
    # Manual formatting as fallback
    print(f"""
    Job Title: {result.get('position', 'Not specified')}
    Company: {result.get('company_name', 'Not specified')}
    
    Requirements:
    {chr(10).join(f'    • {req}' for req in result.get('requirements', []))}
    
    Benefits:
    {chr(10).join(f'    • {benefit}' for benefit in result.get('benefits', []))}
    
    Contact: {result.get('contact_info', 'Not specified')}
    """)

Extracting job information...
Raw extraction: ```json
{
  "company_name": "EmeretusInc",
  "position": "Senior Software Engineer",
  "requirements": [
    "5+ years Python",
    "AWS experience",
    "Kubernetes",
    "Docker"
  ],
  "benefits": [
    "Health insurance",
    "401k matching",
    "flexible hours",
    "remote work options"
  ],
  "contact_info": "careers@emeretus.com or call 555-0123"
}
```
Cleaned extraction: ```json
{
  "company_name": "Emeretus Inc",
  "position": "Senior Software Engineer",
  "requirements": [
    "5+ years Python",
    "AWS experience",
    "Kubernetes",
    "Docker"
  ],
  "benefits": [
    "Health insurance",
    "401k matching",
    "Flexible hours",
    "Remote work options"
  ],
  "contact_info": "careers@emeretus.com or call 555-0123",
  "location": "Not specified",
  "employment_type": "Not specified",
  "job_description": "Not specified"
}
```

Extraction completed successfully!

Extracted Job Data:
{
  "company_name": "Emeretus Inc",
  "po

## 2. Batch Processing - Efficient handling of multiple items__


In [9]:
import asyncio
from tqdm import tqdm

# Batch processing for multiple items with progress
sentiment_prompt = ChatPromptTemplate.from_template("""
Analyze the sentiment of this product review and classify as: Positive, Negative, or Neutral.

Review: {review}

Sentiment analysis:
""")

aspect_prompt = ChatPromptTemplate.from_template("""
Extract key aspects mentioned in this product review:

Review: {review}

Key aspects (comma-separated):
""")

summary_prompt = ChatPromptTemplate.from_template("""
Summarize this product review in one sentence:

Review: {review}

Summary:
""")

# Create analysis chains
sentiment_chain = sentiment_prompt | model | StrOutputParser()
aspect_chain = aspect_prompt | model | StrOutputParser()
summary_chain = summary_prompt | model | StrOutputParser()

# Parallel analysis for single review
single_analysis = RunnableParallel({
    "sentiment": sentiment_chain,
    "aspects": aspect_chain,
    "summary": summary_chain
})

# Batch processing function
def batch_analyze_reviews(reviews):
    """Process multiple reviews with progress tracking"""
    results = []
    
    for review in tqdm(reviews, desc="Analyzing reviews"):
        try:
            analysis = single_analysis.invoke({"review": review})
            analysis["original_review"] = review
            results.append(analysis)
        except Exception as e:
            print(f"Error processing review: {e}")
            continue
    
    return results

# Aggregate results
aggregation_prompt = ChatPromptTemplate.from_template("""
Based on the analysis of {count} product reviews, provide overall insights:

Review Analyses:
{analyses}

Overall Insights:
- Overall sentiment distribution
- Most mentioned aspects
- Common themes in positive reviews
- Common issues in negative reviews

Comprehensive Analysis:
""")

def create_aggregation_report(analyses):
    '''Create a comprehensive report from individual review analyses.'''


    analysis_text = "\n\n".join([
        f"Review {i+1}: {a['summary']} | Sentiment: {a['sentiment']} | Aspects: {a['aspects']}"
        for i, a in enumerate(analyses)
    ])
    
    return aggregation_prompt | model | StrOutputParser()

# Test with sample reviews
sample_reviews = [
    "The product is amazing! Great quality and fast shipping. Definitely recommend!",
    "Poor quality, arrived damaged. Customer service was unhelpful.",
    "It's okay for the price but nothing special. Does what it's supposed to.",
    "Excellent product! Exceeded my expectations. The features are very useful.",
    "Not worth the money. Broke after two weeks of light use.",
    "Good value overall. Some minor issues but generally satisfied.",
]

print("Processing batch of reviews...")
individual_analyses = batch_analyze_reviews(sample_reviews)

print("\nIndividual Analyses:")
for i, analysis in enumerate(individual_analyses):
    print(f"\nReview {i+1}:")
    print(f"  Summary: {analysis['summary']}")
    print(f"  Sentiment: {analysis['sentiment']}")
    print(f"  Aspects: {analysis['aspects']}")

# Generate overall report
aggregate_chain = create_aggregation_report(individual_analyses)
overall_report = aggregate_chain.invoke({
    "count": len(individual_analyses),
    "analyses": "\n".join([f"Review {i+1}: {a['summary']} | {a['sentiment']} | {a['aspects']}" 
                          for i, a in enumerate(individual_analyses)])
})

print("\n" + "="*60)
print("OVERALL ANALYSIS REPORT")
print("="*60)
print(overall_report)

Processing batch of reviews...


Analyzing reviews: 100%|██████████| 6/6 [00:08<00:00,  1.46s/it]



Individual Analyses:

Review 1:
  Summary: The reviewer praises the product for its excellent quality and quick shipping, highly recommending it.
  Sentiment: The sentiment of the product review is Positive. The use of phrases like "amazing," "great quality," and "definitely recommend" indicates a favorable opinion of the product.
  Aspects: amazing product, great quality, fast shipping, highly recommended

Review 2:
  Summary: The product was of poor quality, arrived damaged, and customer service was unhelpful.
  Sentiment: The sentiment of the product review is **Negative**. The reviewer expresses dissatisfaction with the quality of the product, mentions it arrived damaged, and indicates that customer service did not provide assistance.
  Aspects: poor quality, arrived damaged, unhelpful customer service

Review 3:
  Summary: The product is adequate for its price but lacks any standout features.
  Sentiment: The sentiment of the product review is Neutral. The reviewer acknowledges t